In [2]:
from collections import Counter

def compute_tf(document):
    return Counter(document)

In [3]:
document = ["information", "retrieval", "retrieval", "system"]

result = compute_tf(document)

print(result)

Counter({'retrieval': 2, 'information': 1, 'system': 1})


In [4]:
documents = {
    "D1": ["information", "retrieval", "retrieval", "system"],
    "D2": ["information", "system", "system", "system"],
    "D3": ["retrieval", "model", "system"],
    "D4": ["information", "retrieval", "model", "model"]
}

vocabulary = ["information", "retrieval", "system", "model"]

In [5]:
import pandas as pd

def build_tf_matrix(documents, vocabulary):

    matrix = {}

    for doc_name, document in documents.items():

        tf_counts = compute_tf(document)

        matrix[doc_name] = [
            tf_counts.get(term, 0)
            for term in vocabulary
        ]

    return pd.DataFrame(
        matrix,
        index=vocabulary
    ).T

In [6]:
tf_matrix = build_tf_matrix(documents, vocabulary)

display(tf_matrix)

,information,retrieval,system,model
D1,1,2,1,0
D2,1,0,3,0
D3,0,1,1,1
D4,1,1,0,2


In [7]:
def compute_normalized_tf(document):

    tf = compute_tf(document)

    total = len(document)

    normalized_tf = {}

    for term in vocabulary:
        normalized_tf[term] = tf.get(term, 0) / total

    return normalized_tf

In [8]:
normalized_matrix = {}

for doc_name, document in documents.items():

    normalized_matrix[doc_name] = compute_normalized_tf(document)

normalized_tf_matrix = pd.DataFrame(normalized_matrix).T

display(normalized_tf_matrix.round(4))

,information,retrieval,system,model
D1,0.25,0.5000,0.2500,0.0000
D2,0.25,0.0000,0.7500,0.0000
D3,0.00,0.3333,0.3333,0.3333
D4,0.25,0.2500,0.0000,0.5000


In [9]:
for doc_name, document in documents.items():

    tf = compute_tf(document)

    max_frequency = max(tf.values())

    print(doc_name, ":")

    for term in vocabulary:

        if tf.get(term, 0) == max_frequency:
            print(term, "=", max_frequency)

D1 :
retrieval = 2
D2 :
system = 3
D3 :
retrieval = 1
system = 1
model = 1
D4 :
model = 2


In [10]:
def compute_df(documents, vocabulary):

    df = {}

    for term in vocabulary:

        count = 0

        for document in documents.values():

            if term in document:
                count = count + 1

        df[term] = count

    return df

In [11]:
df = compute_df(documents, vocabulary)

print(df)

{'information': 3, 'retrieval': 3, 'system': 3, 'model': 2}


In [23]:
import math

def compute_idf(df, n_documents):
    idf = {}
    for term in df:
        idf[term] = math.log10(n_documents / df[term])
    return idf

def compute_idf_smooth(df, n_documents):
    idf_smooth = {}
    for term in df:
        idf_smooth[term] = math.log10(1 + n_documents / df[term])
    return idf_smooth

In [24]:
N = len(documents)

idf1 = compute_idf(df, N)

idf2 = compute_idf_smooth(df, N)

print("IDF-1:")
print(idf1)

print("IDF-2:")
print(idf2)

IDF-1:
{'information': 0.12493873660829992, 'retrieval': 0.12493873660829992, 'system': 0.12493873660829992, 'model': 0.3010299956639812}
IDF-2:
{'information': 0.3679767852945943, 'retrieval': 0.3679767852945943, 'system': 0.3679767852945943, 'model': 0.47712125471966244}


In [25]:
df_idf_table = pd.DataFrame({
    "Term": vocabulary,
    "DF": [df[term] for term in vocabulary],
    "IDF-1": [idf1[term] for term in vocabulary],
    "IDF-2": [idf2[term] for term in vocabulary]
})

display(df_idf_table.round(4))

,Term,DF,IDF-1,IDF-2
0,information,3,0.1249,0.3680
1,retrieval,3,0.1249,0.3680
2,system,3,0.1249,0.3680
3,model,2,0.3010,0.4771


In [26]:
# IDF-1
print("IDF-1")

max_idf1 = max(idf1.values())
min_idf1 = min(idf1.values())

print("Highest IDF:")
for term in vocabulary:
    if idf1[term] == max_idf1:
        print(term)

print("Lowest IDF:")
for term in vocabulary:
    if idf1[term] == min_idf1:
        print(term)


# IDF-2
print("IDF-2")

max_idf2 = max(idf2.values())
min_idf2 = min(idf2.values())

print("Highest IDF:")
for term in vocabulary:
    if idf2[term] == max_idf2:
        print(term)

print("Lowest IDF:")
for term in vocabulary:
    if idf2[term] == min_idf2:
        print(term)

IDF-1
Highest IDF:
model
Lowest IDF:
information
retrieval
system
IDF-2
Highest IDF:
model
Lowest IDF:
information
retrieval
system


In [28]:
N = len(documents)

found = False

for term in vocabulary:
    if df[term] == N:
        print("Term:", term)
        print("DF:", df[term])
        print("IDF-1:", idf1[term])
        print("IDF-2:", idf2[term])
        found = True

if found == False:
    print("No term appears in all documents.")

No term appears in all documents.


In [29]:
def compute_tfidf(tf_matrix, idf):
    tfidf_matrix = tf_matrix.copy()

    for term in vocabulary:
        tfidf_matrix[term] = tf_matrix[term] * idf[term]

    return tfidf_matrix

In [30]:
tfidf_matrix = compute_tfidf(tf_matrix, idf1)

display(tfidf_matrix.round(4))

,information,retrieval,system,model
D1,0.1249,0.2499,0.1249,0.0000
D2,0.1249,0.0000,0.3748,0.0000
D3,0.0000,0.1249,0.1249,0.3010
D4,0.1249,0.1249,0.0000,0.6021


In [31]:
tfidf_matrix = {}

for doc_name in documents:
    tfidf_matrix[doc_name] = {}

    for term in vocabulary:
        tf = tf_matrix.loc[doc_name, term]
        idf = idf1[term]

        tfidf_matrix[doc_name][term] = tf * idf

tfidf_matrix = pd.DataFrame(tfidf_matrix).T

display(tfidf_matrix.round(4))

,information,retrieval,system,model
D1,0.1249,0.2499,0.1249,0.0000
D2,0.1249,0.0000,0.3748,0.0000
D3,0.0000,0.1249,0.1249,0.3010
D4,0.1249,0.1249,0.0000,0.6021


In [32]:
normalized_tfidf_matrix = {}

for doc_name in documents:
    normalized_tfidf_matrix[doc_name] = {}

    for term in vocabulary:
        tf = normalized_tf_matrix.loc[doc_name, term]
        idf = idf1[term]

        normalized_tfidf_matrix[doc_name][term] = tf * idf

normalized_tfidf_matrix = pd.DataFrame(normalized_tfidf_matrix).T

display(normalized_tfidf_matrix.round(4))

,information,retrieval,system,model
D1,0.0312,0.0625,0.0312,0.0000
D2,0.0312,0.0000,0.0937,0.0000
D3,0.0000,0.0416,0.0416,0.1003
D4,0.0312,0.0312,0.0000,0.1505


In [33]:
for doc_name in documents:

    print("Document:", doc_name)

    # Highest term in raw TF-IDF
    max_raw = tfidf_matrix.loc[doc_name].max()

    for term in vocabulary:
        if tfidf_matrix.loc[doc_name, term] == max_raw:
            print("Raw TF-IDF:", term)

    # Highest term in normalized TF-IDF
    max_normalized = normalized_tfidf_matrix.loc[doc_name].max()

    for term in vocabulary:
        if normalized_tfidf_matrix.loc[doc_name, term] == max_normalized:
            print("Normalized TF-IDF:", term)

    print("------------------")

Document: D1
Raw TF-IDF: retrieval
Normalized TF-IDF: retrieval
------------------
Document: D2
Raw TF-IDF: system
Normalized TF-IDF: system
------------------
Document: D3
Raw TF-IDF: model
Normalized TF-IDF: model
------------------
Document: D4
Raw TF-IDF: model
Normalized TF-IDF: model
------------------


In [35]:
tfidf_idf2 = tf_matrix.copy()

for term in vocabulary:
    tfidf_idf2[term] = tf_matrix[term] * idf2[term]

display(tfidf_idf2.round(4))

,information,retrieval,system,model
D1,0.368,0.736,0.3680,0.0000
D2,0.368,0.000,1.1039,0.0000
D3,0.000,0.368,0.3680,0.4771
D4,0.368,0.368,0.0000,0.9542


In [39]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [" ".join(doc) for doc in documents.values()]

vectorizer = TfidfVectorizer(
    vocabulary=vocabulary,
    norm=None,
    smooth_idf=False
)

result = vectorizer.fit_transform(corpus)

sklearn_matrix = pd.DataFrame(
    result.toarray(),
    index=documents.keys(),
    columns=vocabulary
)

display(sklearn_matrix.round(4))

,information,retrieval,system,model
D1,1.2877,2.5754,1.2877,0.0000
D2,1.2877,0.0000,3.8630,0.0000
D3,0.0000,1.2877,1.2877,1.6931
D4,1.2877,1.2877,0.0000,3.3863


In [37]:
print(vectorizer.idf_)

[1.28768207 1.28768207 1.28768207 1.69314718]


In [40]:
print("Manual TF-IDF:")
display(tfidf_matrix.round(4))

print("Scikit-learn TF-IDF:")
display(sklearn_matrix.round(4))

Manual TF-IDF:


,information,retrieval,system,model
D1,0.1249,0.2499,0.1249,0.0000
D2,0.1249,0.0000,0.3748,0.0000
D3,0.0000,0.1249,0.1249,0.3010
D4,0.1249,0.1249,0.0000,0.6021


Scikit-learn TF-IDF:


,information,retrieval,system,model
D1,1.2877,2.5754,1.2877,0.0000
D2,1.2877,0.0000,3.8630,0.0000
D3,0.0000,1.2877,1.2877,1.6931
D4,1.2877,1.2877,0.0000,3.3863


In [41]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(vocabulary=vocabulary)
result = cv.fit_transform(corpus)

display(pd.DataFrame(
    result.toarray(),
    index=documents.keys(),
    columns=vocabulary
))

,information,retrieval,system,model
D1,1,2,1,0
D2,1,0,3,0
D3,0,1,1,1
D4,1,1,0,2
